In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Import pandas and load the CSV files into separate DataFrames.
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/DSH/Lab-gruppo/Lab2/features_CS_binWidth10_with_class.csv")
df2 = pd.read_csv("/content/drive/MyDrive/DSH/Lab-gruppo/Lab2/features_dataset2.csv")

In [4]:

import numpy as np
import pandas as pd

# === Funzione per ICC(3,1) ===
def icc_3_1(data2d: np.ndarray) -> float:
    """
    ICC(3,1): two-way mixed, single-measure, absolute agreement.
    data2d: array (n_subjects x k_raters)
    """
    n, k = data2d.shape
    if k < 2 or n < 2:
        return np.nan

    mean_rater = data2d.mean(axis=0, keepdims=True)         # (1, k)
    mean_subject = data2d.mean(axis=1, keepdims=True)       # (n, 1)
    grand_mean  = data2d.mean()

    # Mean squares
    MSR = (k * ((mean_subject - grand_mean) ** 2).sum()) / (n - 1)
    SSE = ((data2d - mean_subject - mean_rater + grand_mean) ** 2).sum()
    MSE = SSE / ((n - 1) * (k - 1))

    return (MSR - MSE) / (MSR + (k - 1) * MSE)


# === Aggrega se ci sono più righe per paziente ===
df_agg  = df.groupby("ID", as_index=False).mean(numeric_only=True)
df2_agg = df2.groupby("ID", as_index=False).mean(numeric_only=True)

# === Seleziona solo colonne numeriche e rimuovi ID / Class ===
num_df  = df_agg.select_dtypes(include=np.number)
num_df2 = df2_agg.select_dtypes(include=np.number)

feat_df  = [c for c in num_df.columns  if c not in ["ID", "DEFINITIVE DIAGNOSIS "]]
feat_df2 = [c for c in num_df2.columns if c not in ["ID", "DEFINITIVE DIAGNOSIS "]]

# === Trova le feature in comune ===
common_features = sorted(set(feat_df).intersection(feat_df2))
if not common_features:
    raise ValueError("Nessuna feature numerica in comune tra df e df2.")

# === Allinea i pazienti ===
left  = num_df[["ID"] + common_features].copy()
right = num_df2[["ID"] + common_features].copy()

aligned = left.merge(right, on="ID", how="inner", suffixes=("_seg1", "_seg2"))
if aligned.empty:
    raise ValueError("Nessun paziente in comune tra df e df2 dopo il merge sugli ID.")

# === Calcola ICC(3,1) per ogni feature ===
rows = []
for f in common_features:
    mat = aligned[[f + "_seg1", f + "_seg2"]].to_numpy(dtype=float)
    mask = np.isfinite(mat).all(axis=1)
    mat = mat[mask]
    n = mat.shape[0]
    icc_val = icc_3_1(mat) if n >= 3 else np.nan

    rows.append({
        "feature": f,
        "n_subjects": n,
        "ICC(3,1)": icc_val
    })

# === Risultati ===
icc_df = pd.DataFrame(rows).sort_values("ICC(3,1)", ascending=False)
icc_df["Robustness"] = pd.cut(
    icc_df["ICC(3,1)"],
    bins=[-np.inf, 0.5, 0.75, 0.9, np.inf],
    labels=["Poor", "Moderate", "Good", "Excellent"]
)

# === Salva o visualizza ===
icc_df.to_csv("/content/drive/MyDrive/DSH/Lab-gruppo/Lab2/icc_feature_robustness.csv", index=False)